In [ ]:
# ipython -c "%run plot_tail.ipynb"

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import style
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Paper specific settings
STANDARD_WIDTH = 17.8
SINGLE_COL_WIDTH = STANDARD_WIDTH / 2
DOUBLE_COL_WIDTH = STANDARD_WIDTH
def cm_to_inch(value):
    return value / 2.54

# Match the OSDI-Pa plotting convention.
matplotlib.rcParams['text.usetex'] = False
style.use('bmh')
plt.rcParams['axes.grid'] = True
plt.rcParams['axes.grid.axis'] = 'y'
plt.rcParams['grid.linewidth'] = 0.5
plt.rcParams['hatch.linewidth'] = 0.5
plt.rcParams['font.family'] = 'Nimbus Roman'
plt.rcParams['grid.linestyle'] = '--'

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
with open(RESULTS / 'robustness.json', 'r', encoding='utf-8') as handle:
    runtime_rows = json.load(handle)
with open(RESULTS / 'real_agent_robustness.json', 'r', encoding='utf-8') as handle:
    real_agent = json.load(handle)
runtime = pd.DataFrame([row for row in runtime_rows if row.get('suite') == 'p50_p95'])
runtime = runtime.set_index('mode').loc[['agenttx_without_read_tracing', 'agenttx_full']].reset_index()

fig = plt.figure(dpi=300, figsize=(cm_to_inch(DOUBLE_COL_WIDTH), cm_to_inch(5.2)))
cmap = 'tab10'

# (a) Runtime tail latency: median versus the 95th percentile.
ax0 = plt.subplot(1, 2, 1)
x = np.arange(len(runtime))
bar_width = 0.34
p50 = runtime['step_p50_ms'].astype(float).to_numpy()
p95 = runtime['step_p95_ms'].astype(float).to_numpy()
ax0.bar(x - bar_width / 2, p50, width=bar_width, color=plt.get_cmap(cmap)(0), hatch='///', linewidth=0.5, label='p50')
ax0.bar(x + bar_width / 2, p95, width=bar_width, color=plt.get_cmap(cmap)(3), linewidth=0.5, label='p95')
ax0.set_xticks(x, labels=['AgentTX\nno-trace', 'AgentTX\nfull'], fontsize=7)
ax0.set_ylabel('Per-call latency (ms)', fontsize=8)
ax0.set_title('(a) Deterministic workload tail', fontsize=8)
ax0.tick_params(bottom=False, top=False, left=False, right=False)
ax0.tick_params(axis='y', labelsize=8)
ax0.legend(loc='upper left', fontsize=6, frameon=False, handlelength=1.2, columnspacing=0.5)

# (b) Real-agent task latency and tool-call tail.
ax1 = plt.subplot(1, 2, 2)
metrics = ['wall_p50_s', 'wall_p95_s']
values = [float(real_agent[metric]) for metric in metrics]
bars = ax1.bar(np.arange(2), values, width=0.55, color=[plt.get_cmap(cmap)(2), plt.get_cmap(cmap)(4)], linewidth=0.5, hatch=['///', ''])
ax1.set_xticks(np.arange(2), labels=['wall p50', 'wall p95'], fontsize=7)
ax1.set_ylabel('Task latency (s)', fontsize=8)
ax1.set_title('(b) Real-agent refactor', fontsize=8)
ax1.tick_params(bottom=False, top=False, left=False, right=False)
ax1.tick_params(axis='y', labelsize=8)
for bar, value in zip(bars, values):
    ax1.text(bar.get_x() + bar.get_width() / 2, value + max(values) * 0.03, f'{value:.2f}', ha='center', va='bottom', fontsize=7)
ax1.text(0.5, 0.92, f"success={real_agent['success_rate']:.0%}", transform=ax1.transAxes, ha='center', fontsize=7)

for ax in fig.axes:
    for axis in ['top', 'bottom', 'left', 'right']:
        ax.spines[axis].set_linewidth(0.5)
plt.tight_layout(pad=0.4)
plt.savefig(ROOT / 'motivation' / 'FIG-Motivation-Tail.pdf', bbox_inches='tight', pad_inches=0)
plt.show()
